In [6]:
data_path = "../raw/"

datasets = {
    "contracts": pd.read_csv(data_path + "contracts_raw.csv"),
    "countries": pd.read_csv(data_path + "countries_raw.csv"),
    "education": pd.read_csv(data_path + "education_raw.csv"),
    "loans":     pd.read_csv(data_path + "loans_raw.csv")
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

contracts: 273504 rows, 20 columns
countries: 296 rows, 10 columns
education: 3000 rows, 9 columns
loans: 11236 rows, 30 columns


In [14]:
for name, df in datasets.items():
    print(f"\n{name}:")
    print(df.columns.tolist())


contracts:
['as_of_date', 'borrower_contract_reference_number', 'borrower_country', 'borrower_country_code', 'contract_description', 'contract_signing_date', 'fiscal_year', 'procurement_category', 'procurement_method', 'project_global_practice', 'project_id', 'project_name', 'region', 'review_type', 'supplier', 'supplier_contract_amount_usd', 'supplier_country', 'supplier_country_code', 'supplier_id', 'wb_contract_number']

countries:
['id', 'iso2Code', 'name', 'region', 'adminregion', 'incomeLevel', 'lendingType', 'capitalCity', 'longitude', 'latitude']

education:
['indicator', 'country', 'countryiso3code', 'date', 'value', 'unit', 'obs_status', 'decimal', 'indicator_code']

loans:
['agreement_signing_date', 'board_approval_date', 'borrower', 'borrowers_obligation_us_', 'cancelled_amount_us_', 'closed_date_most_recent', 'country', 'country_code', 'credit_number', 'credit_status', 'credits_held_us_', 'currency_of_commitment', 'disbursed_amount_us_', 'due_3rd_party_us_', 'due_to_ida_u

In [17]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"TABLE: {name.upper()}")
    print(f"{'='*50}")
    
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    
    info = pd.DataFrame({
        "dtype":       df.dtypes,
        "missing":     missing,
        "missing_%":   missing_pct,
        "sample_val":  df.iloc[0]
    })
    
    print(info.to_string())


TABLE: CONTRACTS
                                      dtype  missing  missing_%                                                                sample_val
as_of_date                           object        0        0.0                                                               21-Apr-2026
borrower_contract_reference_number   object        1        0.0                                                    UDP PPG-QCBS-2016-4-12
borrower_country                     object        0        0.0                                                           Kyrgyz Republic
borrower_country_code                object     9018        3.3                                                                        KG
contract_description                 object      347        0.1  Technical Supervision of construction in pilot Schools and Kindergartens
contract_signing_date                object        0        0.0                                                               19-May-2022
fiscal_year     

In [20]:
contracts_clean = datasets["contracts"].copy()

dups = contracts_clean.duplicated().sum()
print(f"Duplicate rows: {dups}")
dups_contract = contracts_clean.duplicated(subset=["wb_contract_number"]).sum()
print(f"Duplicate wb_contract_number: {dups_contract}")

Duplicate rows: 0
Duplicate wb_contract_number: 11098


In [22]:
dups_combo = contracts_clean.duplicated(subset=["wb_contract_number", "supplier_id"]).sum()
print(f"Duplicate (wb_contract_number + supplier_id): {dups_combo}")

Duplicate (wb_contract_number + supplier_id): 0


In [25]:
for col in ["as_of_date", "contract_signing_date"]:
    contracts_clean[col] = pd.to_datetime(contracts_clean[col], format="%d-%b-%Y", errors="coerce")

print(contracts_clean[["as_of_date", "contract_signing_date"]].dtypes)
print(contracts_clean[["as_of_date", "contract_signing_date"]].head(3))


as_of_date               datetime64[ns]
contract_signing_date    datetime64[ns]
dtype: object
  as_of_date contract_signing_date
0 2026-04-21            2022-05-19
1 2026-04-21            2022-05-19
2 2026-04-21            2020-12-23


In [26]:
nulls = contracts_clean.isnull().sum()
print(nulls[nulls > 0])

borrower_contract_reference_number        1
borrower_country_code                  9018
contract_description                    347
project_global_practice               24456
review_type                              36
supplier_contract_amount_usd              6
supplier_country                          6
supplier_country_code                    23
supplier_id                               6
dtype: int64


In [27]:
contracts_clean = contracts_clean[contracts_clean["borrower_contract_reference_number"].notna()]

contracts_clean = contracts_clean.dropna(subset=["supplier_contract_amount_usd", 
                                                   "supplier_country", 
                                                   "supplier_id"])

contracts_clean["contract_description"]   = contracts_clean["contract_description"].fillna("Unknown")
contracts_clean["project_global_practice"] = contracts_clean["project_global_practice"].fillna("Unknown")
contracts_clean["review_type"]             = contracts_clean["review_type"].fillna("Unknown")

code_lookup = contracts_clean.dropna(subset=["supplier_country_code"])\
                              .drop_duplicates(subset=["supplier_country"])\
                              .set_index("supplier_country")["supplier_country_code"]

contracts_clean["supplier_country_code"] = contracts_clean.apply(
    lambda row: code_lookup.get(row["supplier_country"], row["supplier_country_code"])
    if pd.isna(row["supplier_country_code"]) else row["supplier_country_code"],
    axis=1
)

print(f"Shape after cleaning: {contracts_clean.shape}")
print(f"\nRemaining nulls:")
print(contracts_clean.isnull().sum()[contracts_clean.isnull().sum() > 0])

Shape after cleaning: (273497, 20)

Remaining nulls:
borrower_country_code    9018
supplier_country_code      17
dtype: int64


In [29]:
contracts_clean["supplier_country_code"] = contracts_clean["supplier_country_code"].fillna("Unknown")

print(f"Final shape: {contracts_clean.shape}")
print(f"Remaining nulls:\n{contracts_clean.isnull().sum()[contracts_clean.isnull().sum() > 0]}")
print(f"\nDate ranges:")
print(f"contract_signing_date: {contracts_clean['contract_signing_date'].min()} : {contracts_clean['contract_signing_date'].max()}")

Final shape: (273497, 20)
Remaining nulls:
borrower_country_code    9018
dtype: int64

Date ranges:
contract_signing_date: 2019-07-01 00:00:00 : 2026-04-19 00:00:00


In [30]:
contracts_clean.to_csv("../cleaned/contracts_clean.csv", index=False)
print("Saved: ../cleaned/contracts_clean.csv")

Saved: ../cleaned/contracts_clean.csv
